# CDM: explainable collateral for a margin obligation

**Business question:** Can the posted collateral cover a USD 1,000,000 margin obligation? What changes when a downgrade effective Tuesday is only recorded on Wednesday, and which unencumbered lot could replace it?

### General motivation

Lending requires collateral: if you borrow money or hold a derivatives position, the other side asks you to post assets (cash, bonds) it can claim if you default. That safety check is repeated daily, so both sides need the same answer to "are we covered?"

There is complexity because the answer depends on many scattered facts: which assets the agreement allows, what each asset is worth after a safety discount, whether an asset is already pledged elsewhere, whether its credit rating still qualifies, and whether the information arrived late. There are potential inconsistencies: a rating can be true on Tuesday but only recorded on Wednesday, a missing rating is not the same as a bad rating, and two snapshots of history can give different coverage for the same day.

### Glossary for non-experts

- **Margin obligation:** how much collateral you must hold (here: USD 1,000,000).
- **Collateral lot:** one deposited asset, e.g. "Bond A" or "USD cash".
- **Eligible collateral schedule:** the agreement's allow-list (here a real FINOS CDM `EligibleCollateralSpecification` object): which currencies and asset types count.
- **Haircut:** a safety discount on market value (e.g. 5% off bonds, 0% off cash). GestaltDB does not price assets; the app computes discounted "credit" first.
- **Encumbered:** already pledged to someone else, so it cannot be reused.
- **Rating:** an external opinion of credit quality. Debt needs positive investment-grade evidence here; cash does not.
- **Valid time vs. recording time:** when something was true in the business world vs. when we learned it in the database. A Tuesday downgrade recorded Wednesday changes Tuesday's restated answer but not Monday's.
- **Coverage / shortfall:** discounted value of eligible posted lots vs. the obligation; shortfall is what is still missing.

### Problem statement in plain English

We hold a few lots against a USD 1,000,000 obligation. Initially Bond A plus cash look sufficient. Then a late feed says Bond A lost its qualifying rating effective Tuesday. We must restate Tuesday's coverage, keep Monday's answer unchanged, prove why each lot passes or fails, and find one unencumbered replacement that closes the gap — without rewriting history or silently guessing.

### What we will do using GestaltDB

1. Load a real CDM schedule object with an explicit, narrow local policy on top.
2. Store the fixed inventory as a property graph with typed relationships.
3. Filter, project, and structurally validate the graph into reasoning facts.
4. Run explainable eligibility rules with proofs, then sum eligible posted credit.
5. Replay business history across recording horizons (pre-feed vs. restated).
6. Propose a hypothetical substitute and check coverage without mutating history.

This standalone, synthetic example starts from actual FINOS CDM `EligibleCollateralSpecification` objects—not the schema-exploration notebook. It demonstrates an explicit CDM-to-property-graph mapping, indexed filtering, typed projection/validation/reasoning, aggregate explanations, historical claim replay, and offline visualization. No live feeds, credentials, prior notebooks, or repository imports are needed.

**Scope:** a pedagogical policy subset, not the CDM `CheckEligibilityByDetails` implementation, legal advice, a production margin system, or an allocation optimizer. The margin obligation is an application record (not a claimed CDM MarginCall class). Only ratings evolve temporally; inventory, prices, the obligation, and schedule are fixed for the experiment. All parties, identifiers, ratings, and amounts are synthetic.

## 1. Isolated installation

Use Python 3.10–3.13. In a fresh Jupyter kernel, uncomment and run the next cell once, **then restart the kernel**. Do not add this checkout's `src/` to `sys.path`: the example intentionally targets the published rc2 API. The version check fails early if a different package shadows it. `finos-cdm` is pinned independently because generated model APIs evolve.

In [ ]:
%pip install 'gestaltdb[lmdb]==0.14.0rc2' 'finos-cdm==7.8.0'

Note: you may need to restart the kernel to use updated packages.


In [1]:
import hashlib
import importlib.metadata as metadata
import json
import sys
from datetime import datetime, timezone
from decimal import Decimal, ROUND_DOWN
from html import escape
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.display import HTML, display
import gestaltdb

assert (3, 10) <= sys.version_info[:2] < (3, 14), 'Use Python 3.10–3.13'
assert metadata.version('gestaltdb') == '0.14.0rc2', 'Install the pinned rc2 and restart the kernel'
assert metadata.version('finos-cdm') == '7.8.0', 'Install the pinned CDM distribution'
assert Path(gestaltdb.__file__).resolve().parent.parent.name != 'src', 'Use the installed release, not a source checkout'
print('GestaltDB:', metadata.version('gestaltdb'), gestaltdb.__file__)
print('FINOS CDM:', metadata.version('finos-cdm'))

# Bundle-first import resolves generated circular type references.
import finos._bundle
from finos.cdm.product.collateral.EligibleCollateralSpecification import EligibleCollateralSpecification
from gestaltdb.graphdb import Edge, GraphDB, Node
from gestaltdb.ontology import TYPE_PREDICATE
from gestaltdb.reasoning import (
    AggregateRule, Aggregation, Comparison, GenericEngine, GenericProgram,
    GenericRule, Literal, ReasoningLimits, TypedAtom, TypedFact,
)
from gestaltdb.reasoning.model import Variable
from gestaltdb.reasoning.adapters.typed_graph import PinnedTypedGraphSource, TypedGraphMapping
from gestaltdb.reasoning.structural import (
    CardinalityBound, DatatypeConstraint, RequiredField, StructuralSuite, validate_structural,
)
from gestaltdb.viz.api import VizOptions, visualize_nodes_edges

def table(rows):
    rows = list(rows)
    if not rows:
        display(HTML('<p>No rows.</p>'))
        return
    columns = list(rows[0])
    heading = ''.join(f'<th>{escape(str(c))}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(r.get(c, "")))}</td>' for c in columns) + '</tr>' for r in rows)
    display(HTML(f'<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>'))

def usd(cents):
    return f'USD {Decimal(cents) / 100:,.2f}'


GestaltDB: 0.14.0rc2 /home/charilaos/Workspace/gestaltdb/.venv-0.14.0rc1/lib/python3.12/site-packages/gestaltdb/__init__.py
FINOS CDM: 7.8.0


## 2. Real CDM schedule, explicit local policy

The schedule includes USD debt securities at a 5% haircut and USD cash at 0%. Pydantic validates generated types/cardinalities; the adapter checks its supported subset explicitly. Full Rune-condition validation is not claimed: `finos-cdm 7.8.0` raises a generated-code `NameError` when recursively validating these criteria. The application adds an explicit policy overlay: debt needs positive investment-grade evidence, remaining maturity must be at most 3,650 days, and lots pledged elsewhere are excluded. These extra conditions are **not silently represented as CDM schedule terms**. No FIBO or external data import is required for this first example.

In [2]:
schedule_payload = {'criteria': [
    {'collateralCriteria': {'AllCriteria': {'allCriteria': [
        {'CurrencyCodeEnum': 'USD'},
        {'AssetType': {'assetType': 'Security', 'securityType': 'Debt'}},
    ]}}, 'treatment': {'isIncluded': True, 'valuationTreatment': {'haircutPercentage': '0.05'}}},
    {'collateralCriteria': {'AllCriteria': {'allCriteria': [
        {'CurrencyCodeEnum': 'USD'}, {'AssetType': {'assetType': 'Cash'}},
    ]}}, 'treatment': {'isIncluded': True, 'valuationTreatment': {'haircutPercentage': '0'}}},
]}
cdm_schedule = EligibleCollateralSpecification.model_validate(schedule_payload)
canonical_schedule = cdm_schedule.model_dump(mode='json', exclude_none=True)
schedule_digest = hashlib.sha256(json.dumps(canonical_schedule, sort_keys=True).encode()).hexdigest()
POLICY_ID = 'synthetic-usd-margin-policy-v1'
MAX_MATURITY_DAYS = 3650
REQUIRED_CENTS = 100_000_000
MARGIN = 'margin:001'
SCHEDULE = 'schedule:001'

# This adapter is deliberately strict: unsupported schedule forms reject, not approximate.
def schedule_terms(specification):
    if specification.model_dump(mode='json', exclude_none=True) != schedule_payload:
        raise ValueError('Only the exact demonstrated schedule subset is supported')
    terms = {}
    for criterion in specification.criteria:
        assert criterion is not None and criterion.treatment.isIncluded
        parts = criterion.collateralCriteria.AllCriteria.allCriteria
        assert len(parts) == 2 and parts[0].CurrencyCodeEnum.value == 'USD'
        asset = parts[1].AssetType
        if asset.assetType.value == 'Cash':
            kind = 'kind:Cash'
        elif asset.assetType.value == 'Security' and asset.securityType.value == 'Debt':
            kind = 'kind:Debt'
        else:
            raise ValueError('Unsupported asset criterion')
        haircut = criterion.treatment.valuationTreatment.haircutPercentage
        assert Decimal('0') <= haircut < Decimal('1') and kind not in terms
        terms[kind] = haircut
    return terms

haircuts = schedule_terms(cdm_schedule)
table({'kind': k, 'haircut': f'{h:.0%}', 'currency': 'USD'} for k, h in haircuts.items())
print('Schedule SHA-256:', schedule_digest)


kind,haircut,currency
kind:Debt,5%,USD
kind:Cash,0%,USD


Schedule SHA-256: c36196114b22ef7a4350b0d1da4c21e29278afc43c8276c01c171ce770a06727


## 3. Synthetic inventory and exact valuation boundary

Initially posted: Bond A worth USD 700,000 (credit USD 665,000) plus USD 400,000 cash. Bond B is an available substitute, not initially posted. Other lots deliberately exercise rejection paths. A missing rating is **unknown**, not a negative claim.

Application code computes haircut-adjusted credit with `Decimal`, rounding **down** to cents. GestaltDB consumes those integer values; it does not price assets. Each lot is indivisible in this example. Currency conversion, partial allocation, price changes, settlement, issuer concentration, and optimization are out of scope.

In [3]:
lots = [
    {'id': 'lot:bond-a', 'name': 'Bond A — initially posted', 'kind': 'kind:Debt', 'currency': 'USD', 'market_cents': 70_000_000, 'maturity_days': 1800, 'posted': True},
    {'id': 'lot:cash', 'name': 'USD cash — initially posted', 'kind': 'kind:Cash', 'currency': 'USD', 'market_cents': 40_000_000, 'maturity_days': 0, 'posted': True},
    {'id': 'lot:bond-b', 'name': 'Bond B — substitute', 'kind': 'kind:Debt', 'currency': 'USD', 'market_cents': 80_000_000, 'maturity_days': 900, 'posted': False},
    {'id': 'lot:encumbered', 'name': 'Bond C — pledged elsewhere', 'kind': 'kind:Debt', 'currency': 'USD', 'market_cents': 90_000_000, 'maturity_days': 900, 'posted': False},
    {'id': 'lot:unknown', 'name': 'Bond D — no rating evidence', 'kind': 'kind:Debt', 'currency': 'USD', 'market_cents': 90_000_000, 'maturity_days': 900, 'posted': False},
    {'id': 'lot:eur', 'name': 'Bond E — wrong currency', 'kind': 'kind:Debt', 'currency': 'EUR', 'market_cents': 90_000_000, 'maturity_days': 900, 'posted': False},
    {'id': 'lot:long', 'name': 'Bond F — too long', 'kind': 'kind:Debt', 'currency': 'USD', 'market_cents': 90_000_000, 'maturity_days': 4000, 'posted': False},
]
for lot in lots:
    # Foreign-currency lots have no USD credit; never imply an FX conversion.
    if lot['currency'] == 'USD':
        lot['credit_cents'] = int((Decimal(lot['market_cents']) * (1 - haircuts[lot['kind']])).quantize(Decimal('1'), rounding=ROUND_DOWN))
lot_by_id = {lot['id']: lot for lot in lots}
table({'lot': x['name'], 'currency': x['currency'], 'credit': usd(x['credit_cents']) if 'credit_cents' in x else 'not converted', 'posted': x['posted']} for x in lots)


lot,currency,credit,posted
Bond A — initially posted,USD,"USD 665,000.00",True
USD cash — initially posted,USD,"USD 400,000.00",True
Bond B — substitute,USD,"USD 760,000.00",False
Bond C — pledged elsewhere,USD,"USD 855,000.00",False
Bond D — no rating evidence,USD,"USD 855,000.00",False
Bond E — wrong currency,EUR,not converted,False
Bond F — too long,USD,"USD 855,000.00",False


In [4]:
# Rerunning this cell closes the old handle before removing only its temporary workspace.
previous_graph = globals().pop('graph', None)
if previous_graph is not None:
    previous_graph.close()
previous_workspace = globals().pop('workspace', None)
if previous_workspace is not None:
    previous_workspace.cleanup()
workspace = TemporaryDirectory(prefix='gestaltdb_cdm_margin_v2_')
workdir = Path(workspace.name)
artifacts = workdir / 'artifacts'
artifacts.mkdir()
graph = GraphDB.create(workdir / 'graph', backend='lmdb', serializer='json')

nodes = [
    Node(MARGIN, labels=['MarginObligation'], properties={'name': 'Synthetic VM obligation', 'currency': 'USD', 'required_cents': REQUIRED_CENTS}),
    Node(SCHEDULE, labels=['CollateralSchedule'], properties={'name': 'CDM USD schedule', 'cdm_type': 'EligibleCollateralSpecification', 'cdm_version': metadata.version('finos-cdm'), 'cdm_payload': canonical_schedule, 'schedule_sha256': schedule_digest, 'policy_id': POLICY_ID}),
    Node('party:poster', labels=['Party'], properties={'name': 'Synthetic Pledgor'}),
    Node('party:receiver', labels=['Party'], properties={'name': 'Synthetic Receiver'}),
    Node('other-lender', labels=['Party'], properties={'name': 'Other lender'}),
    Node('kind:Cash', labels=['AssetClass'], properties={'name': 'Cash'}),
    Node('kind:Debt', labels=['AssetClass'], properties={'name': 'Debt security'}),
]
edges = [
    Edge('governs', MARGIN, SCHEDULE, {'type': 'GOVERNED_BY'}),
    Edge('poster', MARGIN, 'party:poster', {'type': 'OWED_BY'}),
    Edge('receiver', MARGIN, 'party:receiver', {'type': 'OWED_TO'}),
]
for kind, haircut in haircuts.items():
    edges.append(Edge(f'permits:{kind}', SCHEDULE, kind, {'type': 'PERMITS', 'haircut_bps': int(haircut * 10000)}))
for lot in lots:
    properties = {k: v for k, v in lot.items() if k not in ('id', 'kind', 'posted')}
    nodes.append(Node(lot['id'], labels=['CollateralLot'], properties=properties))
    edges.extend([
        Edge(f'kind:{lot["id"]}', lot['id'], lot['kind'], {'type': 'CLASSIFIED_AS'}),
        Edge(f'candidate:{lot["id"]}', MARGIN, lot['id'], {'type': 'CANDIDATE'}),
    ])
    if lot['posted']:
        edges.append(Edge(f'posted:{lot["id"]}', MARGIN, lot['id'], {'type': 'POSTED'}))
edges.append(Edge('encumbrance', 'lot:encumbered', 'other-lender', {'type': 'ENCUMBERED_BY'}))
graph.nodes.put_many(nodes)
graph.edges.put_many(edges)
graph.indexes.create_node_property_index('currency')
graph.indexes.create_node_property_index('maturity_days')
print('Disposable LMDB:', workdir)


Disposable LMDB: /tmp/gestaltdb_cdm_margin_v2_aunint12


## 4. Filter, project, validate

The indexed currency lookup is a **cohort filter**, not a complete eligibility decision. The explicit typed projection below retains the full small graph so rejected cases remain inspectable. LMDB supplies a verifiable current-state snapshot. Structural validation checks only the declared complete classes; this is not full CDM/SHACL conformance. We deliberately remove a required property from a copy to demonstrate a caught defect.

In [5]:
cohort = graph.query(
    'MATCH (a:CollateralLot {currency: $currency}) RETURN a.name AS lot, a.maturity_days AS maturity ORDER BY lot',
    {'currency': 'USD'},
)
assert len(cohort.records) == 6
table(cohort.records)

mapping = TypedGraphMapping(
    labels=[('CollateralLot', 'CollateralLot'), ('MarginObligation', 'MarginObligation')],
    edge_types=[('GOVERNED_BY', 'governedBy'), ('PERMITS', 'permits'), ('CLASSIFIED_AS', 'assetKind'), ('CANDIDATE', 'candidate'), ('POSTED', 'posted'), ('ENCUMBERED_BY', 'encumbered')],
    node_attributes=[('currency', 'currency'), ('required_cents', 'requiredCents'), ('credit_cents', 'creditCents'), ('maturity_days', 'maturityDays')],
    edge_attributes=[('haircut_bps', 'haircutBps')],
)
with graph.current_read_view(require_verifiable=True) as current_view:
    with PinnedTypedGraphSource(current_view, mapping=mapping) as source:
        static_evidence = list(source.iter_evidence())
        static_context = source.context(POLICY_ID)
static_facts = tuple(item.fact for item in static_evidence)

suite = StructuralSuite(
    TYPE_PREDICATE, closed_classes=['CollateralLot', 'MarginObligation'],
    required=[RequiredField('CollateralLot', 'currency'), RequiredField('CollateralLot', 'maturityDays'), RequiredField('MarginObligation', 'requiredCents')],
    cardinalities=[CardinalityBound('CollateralLot', 'currency', 1, 1)],
    datatypes=[DatatypeConstraint('CollateralLot', 'currency', 'text'), DatatypeConstraint('CollateralLot', 'maturityDays', 'integer'), DatatypeConstraint('MarginObligation', 'requiredCents', 'integer')],
)
report = validate_structural(static_facts, suite)
assert report.clean and not report.truncated
broken = [f for f in static_facts if f != TypedFact('requiredCents', [MARGIN, Literal('integer', REQUIRED_CENTS)])]
broken_report = validate_structural(broken, suite)
assert not broken_report.clean and any(v.kind == 'missing-required' for v in broken_report.violations)
print('Projection:', len(static_facts), 'facts; clean input, planted missing amount detected.')


lot,maturity
Bond A — initially posted,1800
Bond B — substitute,900
Bond C — pledged elsewhere,900
Bond D — no rating evidence,900
Bond F — too long,4000
USD cash — initially posted,0


Projection: 52 facts; clean input, planted missing amount detected.


## 5. Positive evidence + scoped absence + aggregate proofs

Debt eligibility requires positive `ratingCategory(asset, investment-grade)` evidence. Cash does not require a rating. The only closed predicate is `encumbered`: the synthetic inventory export is declared complete for the modeled `other-lender` encumbrance within this experiment. In production, do not make that completeness declaration from a partial feed.

An eligible lot must match the obligation's currency and schedule asset class, satisfy the local maturity limit, and have no modeled external pledge. Aggregation sums eligible **posted** lots, not every candidate. We use two explicit stages: stratified eligibility with proofs, then positive aggregation with proofs. The combined negation/aggregate proof path raises `untruncated proofs must cover the result closure` in published rc2 for this fixture. The explanation display links stage-two inputs back to stage-one witnesses; neither trace alone proves the whole pipeline. Aggregate heads are sinks: coverage/shortfall arithmetic and substitute sizing are performed explicitly in Python.

In [6]:
m, a, s, k, c, d, v, total = [Variable(x) for x in ('m', 'a', 's', 'k', 'c', 'd', 'v', 'total')]
program = GenericProgram([
    GenericRule('cash-needs-no-rating', [TypedAtom('assetKind', [a, 'kind:Cash'])], [], TypedAtom('ratingAllowed', [a])),
    GenericRule('debt-needs-positive-investment-grade', [TypedAtom('assetKind', [a, 'kind:Debt']), TypedAtom('ratingCategory', [a, 'investment-grade'])], [], TypedAtom('ratingAllowed', [a])),
    GenericRule('eligible-for-margin', [
        TypedAtom('candidate', [m, a]), TypedAtom('governedBy', [m, s]),
        TypedAtom('assetKind', [a, k]), TypedAtom('permits', [s, k]),
        TypedAtom('currency', [m, c]), TypedAtom('currency', [a, c]),
        TypedAtom('maturityDays', [a, d]), TypedAtom('ratingAllowed', [a]),
    ], [Comparison('<=', d, Literal('integer', MAX_MATURITY_DAYS))],
       TypedAtom('eligible', [m, a]), negated=[TypedAtom('encumbered', [a, 'other-lender'])]),
    GenericRule('posted-credit-only-if-eligible', [
        TypedAtom('eligible', [m, a]), TypedAtom('posted', [m, a]), TypedAtom('creditCents', [a, v]),
    ], [], TypedAtom('eligiblePostedCredit', [m, a, v])),
], closed=['encumbered'])
coverage_program = GenericProgram([], aggregates=[AggregateRule(
    'total-eligible-posted-credit', [TypedAtom('eligiblePostedCredit', [m, a, v])],
    (m,), (Aggregation('sum', v, total),), TypedAtom('postedCoverageCents', [m, total]),
)])
limits = ReasoningLimits(max_facts=10000, max_proof_nodes=10000, max_proof_edges=20000)
print('Policy program digest:', program.digest)


Policy program digest: sha256:e7c0da8ab63152e0f1903c507308c4a9d318174c4ca28fc7d74573a2ee5bd5c2


## 6. Ratings: business-valid time versus recording history

We use dated business-valid instants, but **real database commit order** for recording history. We do not fake Wednesday wall-clock commits. `before_commit` means the database horizon before the simulated Wednesday feed arrival; `after_commit` means after it.

Rating categories are synthetic source assertions, not probabilities or a built-in rating-scale conversion. This fixture assumes one authoritative category per asset at a valid point; multi-source conflict resolution is out of scope. The explicit bridge below reads only positive entity claims from one source/agent/world/predicate and retains exact claim-version references. It is a small application adapter, not an authenticated combined graph/claim generation: the static snapshot and temporal rating view are separate recorded inputs.

In [7]:
def instant(iso):
    value = datetime.fromisoformat(iso).astimezone(timezone.utc)
    delta = value - datetime(1970, 1, 1, tzinfo=timezone.utc)
    return (delta.days * 86400 + delta.seconds) * 1_000_000 + delta.microseconds

MONDAY = instant('2026-10-19T00:00:00+00:00')
TUESDAY = instant('2026-10-20T00:00:00+00:00')
END = instant('2026-10-23T00:00:00+00:00')
rating_versions = {}
for lot_id in ('lot:bond-a', 'lot:bond-b', 'lot:encumbered', 'lot:eur', 'lot:long'):
    rating_versions[lot_id] = graph.epistemic.assert_claim(
        subject=lot_id, predicate='ratingCategory', object='investment-grade',
        agent='synthetic-rating-feed', source='fixture:ratings-v1', world='collateral-demo',
        polarity='positive', valid=(MONDAY, END),
        provenance={'synthetic': True, 'scenario': 'initial rating export'},
    )
before_commit = max(version.commit_id for version in rating_versions.values())

def rating_inputs(valid_time, horizon):
    with graph.temporal.open_read_view(
        graph, manifest=graph.manifest, database_id=graph.database_id,
        valid_time=valid_time, through_commit=horizon,
    ) as view:
        versions = list(view.iter_claims_as_of(world='collateral-demo', predicate='ratingCategory'))
        selected = [r for r in versions if r.claim.agent == 'synthetic-rating-feed'
                    and r.claim.source == 'fixture:ratings-v1'
                    and r.claim.polarity.value == 'positive' and r.claim.object_kind.value == 'entity']
        facts = tuple(TypedFact('ratingCategory', [r.claim.subject, r.claim.object]) for r in selected)
        evidence = [{'fact': repr(f), 'version_id': r.version_id, 'commit': r.commit_id,
                     'valid_from': r.valid.start.epoch_microseconds, 'valid_to': r.valid.end.epoch_microseconds}
                    for f, r in zip(facts, selected)]
        assert view.commit_horizon == horizon
    return facts, evidence

def evaluate_at(valid_time, horizon, base=static_facts):
    rating_facts, evidence = rating_inputs(valid_time, horizon)
    inputs = tuple(base) + rating_facts
    run = GenericEngine().evaluate_stratified(program, inputs, limits=limits, collect_proofs=True)
    aggregated = GenericEngine().evaluate(coverage_program, run.result.facts, limits=limits, collect_proofs=True)
    assert run.proofs is not None and not run.proofs.truncated
    assert run.proofs.is_current_for(program, inputs)
    assert aggregated.proofs is not None and not aggregated.proofs.truncated
    assert aggregated.proofs.is_current_for(coverage_program, run.result.facts)
    return {'result': aggregated, 'proofs': run.proofs, 'coverage_proofs': aggregated.proofs, 'inputs': inputs, 'rating_evidence': evidence,
            'valid_time': valid_time, 'horizon': horizon}

def eligible_ids(case):
    return {f.values[1] for f in case['result'].facts if f.predicate == 'eligible'}

def coverage(case):
    totals = [f.values[1].value for f in case['result'].facts if f.predicate == 'postedCoverageCents']
    assert len(totals) <= 1
    # No aggregate group exists when no posted lot is eligible: explicit zero default.
    return totals[0] if totals else 0

initial = evaluate_at(TUESDAY, before_commit)
assert eligible_ids(initial) == {'lot:bond-a', 'lot:cash', 'lot:bond-b'}
assert coverage(initial) == 106_500_000
print('Tuesday, before late feed:', usd(coverage(initial)), 'against', usd(REQUIRED_CENTS))


Tuesday, before late feed: USD 1,065,000.00 against USD 1,000,000.00


In [8]:
# Simulated Wednesday arrival: withdraw the old category from Tuesday, then assert the new one.
# A correction valid only on Monday would not remove older support on Tuesday.
# A single transaction/publication batch is not claimed; evaluate only after both writes.
old_a = rating_versions['lot:bond-a']
withdrawn_a = graph.epistemic.retract_claim(
    old_a.claim.claim_id, supersedes_version_id=old_a.version_id, valid=(TUESDAY, END),
    reason='Synthetic late feed: investment-grade assertion no longer applies from Tuesday',
)
downgrade = graph.epistemic.assert_claim(
    subject='lot:bond-a', predicate='ratingCategory', object='below-policy',
    agent='synthetic-rating-feed', source='fixture:ratings-v1', world='collateral-demo',
    polarity='positive', valid=(TUESDAY, END),
    provenance={'synthetic': True, 'scenario': 'simulated Wednesday arrival, effective Tuesday'},
)
after_commit = downgrade.commit_id
restated = evaluate_at(TUESDAY, after_commit)
monday_restated = evaluate_at(MONDAY, after_commit)
historical_replay = evaluate_at(TUESDAY, before_commit)
assert coverage(restated) == 40_000_000
assert coverage(monday_restated) == 106_500_000
assert coverage(historical_replay) == coverage(initial)
assert eligible_ids(restated) == {'lot:cash', 'lot:bond-b'}
assert not initial['proofs'].is_current_for(program, restated['inputs'])

table([
    {'question': 'Tuesday, using pre-arrival history', 'horizon': before_commit, 'coverage': usd(coverage(initial)), 'shortfall': usd(max(0, REQUIRED_CENTS - coverage(initial)))},
    {'question': 'Tuesday, restated after late feed', 'horizon': after_commit, 'coverage': usd(coverage(restated)), 'shortfall': usd(max(0, REQUIRED_CENTS - coverage(restated)))},
    {'question': 'Monday, using corrected history', 'horizon': after_commit, 'coverage': usd(coverage(monday_restated)), 'shortfall': usd(max(0, REQUIRED_CENTS - coverage(monday_restated)))},
])


question,horizon,coverage,shortfall
"Tuesday, using pre-arrival history",5,"USD 1,065,000.00",USD 0.00
"Tuesday, restated after late feed",7,"USD 400,000.00","USD 600,000.00"
"Monday, using corrected history",7,"USD 1,065,000.00",USD 0.00


## 7. Why? Inspect evidence, grounded checks, and aggregate contributors

Each explanation is a bounded selected witness, not every independent justification. `rating_evidence` preserves historical version IDs separately; static projection references remain in `static_evidence`. Neither the custom bridge nor the JSON export below independently authenticates the combined computation.

A missing conclusion must not be presented as a proof of ineligibility. The rejection table gives application diagnostics for this complete fixture, while explanations below prove actual derived facts.

In [9]:
def explanation_nodes(case, fact):
    # Stage two sees the eligibility closure as input. Expand those inputs using stage one.
    trace = case['coverage_proofs'] if fact.predicate == 'postedCoverageCents' else case['proofs']
    explanation = trace.explain(fact)
    assert not explanation.truncated
    merged = {node.fact: node for node in explanation.nodes}
    if trace is case['coverage_proofs']:
        for node in explanation.nodes:
            if node.rule is None:
                earlier = case['proofs'].explain(node.fact)
                assert not earlier.truncated
                for original in earlier.nodes:
                    merged[original.fact] = original
    return tuple(merged[f] for f in sorted(merged, key=repr))

def explain_rows(case, fact):
    return [{'fact': repr(node.fact), 'rule': node.rule.rule_id if node.rule else 'input',
             'comparisons': repr(node.comparisons), 'absence': repr(node.absence),
             'aggregate': repr(node.aggregates)} for node in explanation_nodes(case, fact)]

initial_total = TypedFact('postedCoverageCents', [MARGIN, Literal('integer', coverage(initial))])
table(explain_rows(initial, initial_total))
table(row for row in restated['rating_evidence'] if 'lot:bond-a' in row['fact'])
table([
    {'lot': 'Bond A', 'restated status': 'No positive investment-grade evidence at Tuesday; below-policy category is present'},
    {'lot': 'Bond C', 'restated status': 'Known modeled external encumbrance'},
    {'lot': 'Bond D', 'restated status': 'Unknown rating; not accepted without positive evidence'},
    {'lot': 'Bond E', 'restated status': 'EUR does not match USD; no FX conversion'},
    {'lot': 'Bond F', 'restated status': '4,000 days exceeds local 3,650-day limit'},
])


fact,rule,comparisons,absence,aggregate
"TypedFact(predicate='assetKind', values=('lot:bond-a', 'kind:Debt'))",input,(),(),()
"TypedFact(predicate='assetKind', values=('lot:cash', 'kind:Cash'))",input,(),(),()
"TypedFact(predicate='candidate', values=('margin:001', 'lot:bond-a'))",input,(),(),()
"TypedFact(predicate='candidate', values=('margin:001', 'lot:cash'))",input,(),(),()
"TypedFact(predicate='creditCents', values=('lot:bond-a', Literal(kind='integer', value=66500000)))",input,(),(),()
"TypedFact(predicate='creditCents', values=('lot:cash', Literal(kind='integer', value=40000000)))",input,(),(),()
"TypedFact(predicate='currency', values=('lot:bond-a', Literal(kind='text', value='USD')))",input,(),(),()
"TypedFact(predicate='currency', values=('lot:cash', Literal(kind='text', value='USD')))",input,(),(),()
"TypedFact(predicate='currency', values=('margin:001', Literal(kind='text', value='USD')))",input,(),(),()
"TypedFact(predicate='eligible', values=('margin:001', 'lot:bond-a'))",eligible-for-margin,"(Comparison(op='<=', left=Literal(kind='integer', value=1800), right=Literal(kind='integer', value=3650)),)","(AbsenceCheck(position=0, predicate='encumbered', args=('lot:bond-a', 'other-lender'), scope=('encumbered',)),)",()


fact,version_id,commit,valid_from,valid_to
"TypedFact(predicate='ratingCategory', values=('lot:bond-a', 'below-policy'))",e3852189-806c-48c7-9ac6-ec2abbf5dc5e,7,1792454400000000,1792713600000000


lot,restated status
Bond A,No positive investment-grade evidence at Tuesday; below-policy category is present
Bond C,Known modeled external encumbrance
Bond D,Unknown rating; not accepted without positive evidence
Bond E,EUR does not match USD; no FX conversion
Bond F,"4,000 days exceeds local 3,650-day limit"


## 8. Find a substitute and evaluate a proposal without changing history

The shortfall is USD 600,000. Bond B provides USD 760,000 after haircut. Select only eligible, not-already-posted, indivisible USD lots whose credit covers the shortfall. Sorting by smallest sufficient credit is a transparent demo heuristic, **not** cheapest-to-deliver optimization. Re-evaluate a copied fact set after replacing A with B; this is a hypothetical proposal, not an executed collateral movement or persisted inferred world.

In [10]:
shortfall = max(0, REQUIRED_CENTS - coverage(restated))
already_posted = {f.values[1] for f in static_facts if f.predicate == 'posted'}
substitutes = sorted(
    (lot_by_id[asset] for asset in eligible_ids(restated) - already_posted
     if lot_by_id[asset]['credit_cents'] >= shortfall),
    key=lambda lot: (lot['credit_cents'], lot['id']),
) if shortfall else []
assert [lot['id'] for lot in substitutes] == ['lot:bond-b']
table({'candidate': lot['name'], 'credit': usd(lot['credit_cents']), 'shortfall': usd(shortfall)} for lot in substitutes)
selected = substitutes[0]['id']
proposal_facts = tuple(f for f in static_facts if f != TypedFact('posted', [MARGIN, 'lot:bond-a'])) + (TypedFact('posted', [MARGIN, selected]),)
proposal = evaluate_at(TUESDAY, after_commit, base=proposal_facts)
assert coverage(proposal) == 116_000_000
assert coverage(restated) == 40_000_000  # proposal did not mutate the actual fact set
table([{'proposal': 'Replace Bond A with Bond B; retain cash', 'coverage': usd(coverage(proposal)),
        'obligation': usd(REQUIRED_CENTS), 'surplus': usd(coverage(proposal) - REQUIRED_CENTS)}])


candidate,credit,shortfall
Bond B — substitute,"USD 760,000.00","USD 600,000.00"


proposal,coverage,obligation,surplus
Replace Bond A with Bond B; retain cash,"USD 1,160,000.00","USD 1,000,000.00","USD 160,000.00"


## 9. Offline graph and proof artifacts

These self-contained HTML files need no network at viewing time. The first graph shows the actual fixed inventory, not the hypothetical proposal. The second graph visualizes the initial coverage witness, including aggregate contributor edges. Copy artifacts before the final cleanup cell if you want to retain them.

In [11]:
inventory_figure = visualize_nodes_edges(nodes, edges, options=VizOptions(
    title='Synthetic CDM margin obligation: fixed inventory', height=600,
    node_label_property='name', edge_labels='all',
))
inventory_figure.save(artifacts / 'cdm-margin-inventory.html')
display(inventory_figure)

witness_nodes = explanation_nodes(initial, initial_total)
proof_id = {node.fact: f'proof:{i}' for i, node in enumerate(witness_nodes)}
proof_nodes = [Node(proof_id[node.fact], labels=['ProofFact'], properties={
    'name': repr(node.fact), 'rule': node.rule.rule_id if node.rule else 'input',
}) for node in witness_nodes]
proof_edges = []
for node in witness_nodes:
    supports = list(node.premises)
    for aggregate in node.aggregates:
        supports.extend(aggregate.operands)
    for premise in sorted(set(supports), key=repr):
        assert premise in proof_id
        proof_edges.append(Edge(f'proof-edge:{len(proof_edges)}', proof_id[premise], proof_id[node.fact],
                                {'type': 'SUPPORTS', 'rule': node.rule.rule_id}))
proof_figure = visualize_nodes_edges(proof_nodes, proof_edges, options=VizOptions(
    title='Why initial posted credit totals USD 1,065,000', height=650,
    node_label_property='name', edge_labels='off', label_max_length=65,
))
proof_figure.save(artifacts / 'cdm-margin-coverage-proof.html')
display(proof_figure)

audit = {
    'synthetic': True, 'gestaltdb': metadata.version('gestaltdb'), 'finos_cdm': metadata.version('finos-cdm'),
    'policy_id': POLICY_ID, 'eligibility_program_digest': program.digest, 'coverage_program_digest': coverage_program.digest, 'schedule_sha256': schedule_digest,
    'scope': 'Fixed inventory/prices/schedule; temporal ratings only; custom application bridge',
    'static_context': repr(static_context), 'static_evidence': [repr(e) for e in static_evidence],
    'schedule': canonical_schedule, 'required_cents': REQUIRED_CENTS,
    'before': {'horizon': before_commit, 'valid_time': TUESDAY, 'coverage_cents': coverage(initial), 'rating_evidence': initial['rating_evidence']},
    'restated': {'horizon': after_commit, 'valid_time': TUESDAY, 'coverage_cents': coverage(restated), 'rating_evidence': restated['rating_evidence']},
    'proposal': {'hypothetical': True, 'replace': 'lot:bond-a', 'with': selected, 'coverage_cents': coverage(proposal)},
    'initial_coverage_witness': explain_rows(initial, initial_total),
}
(artifacts / 'cdm-margin-audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
print('Artifacts:', artifacts)


VizFigure(nodes=14, edges=22)

VizFigure(nodes=24, edges=27)

Artifacts: /tmp/gestaltdb_cdm_margin_v2_aunint12/artifacts


## 10. Regression checks

Check half-open validity boundaries, historical replay, missing ratings, zero eligible posted lots, proposal isolation, deterministic closure, and proof freshness. This is correctness evidence for the small scenario—not a scale, durability, financial-model, or security certification.

In [15]:
assert len([f for f in static_facts if f.predicate == TYPE_PREDICATE]) == len(lots) + 1
assert 'lot:bond-a' in eligible_ids(evaluate_at(TUESDAY - 1, after_commit))
assert 'lot:bond-a' not in eligible_ids(evaluate_at(TUESDAY, after_commit))
assert 'lot:unknown' not in eligible_ids(initial)
assert 'lot:unknown' not in eligible_ids(restated)
assert TypedFact('ratingCategory', ['lot:unknown', 'below-policy']) not in restated['inputs']
assert coverage(evaluate_at(TUESDAY, before_commit)) == 106_500_000
no_posted = tuple(f for f in static_facts if f.predicate != 'posted')
assert coverage(evaluate_at(TUESDAY, after_commit, base=no_posted)) == 0
again = evaluate_at(TUESDAY, after_commit)
assert set(again['result'].facts) == set(restated['result'].facts)
assert again['proofs'].horizon == restated['proofs'].horizon
assert not initial['proofs'].is_current_for(program, restated['inputs'])
aggregate_nodes = [n for n in explanation_nodes(initial, initial_total) if n.aggregates]
assert len(aggregate_nodes) == 1 and len(aggregate_nodes[0].aggregates[0].operands) == 2
for asset in ('lot:encumbered', 'lot:eur', 'lot:long'):
    assert asset not in eligible_ids(initial) and asset not in eligible_ids(restated)
unsupported = EligibleCollateralSpecification.model_validate({'criteria': [schedule_payload['criteria'][0]]})
try:
    schedule_terms(unsupported)
except ValueError:
    pass
else:
    raise AssertionError('Unsupported schedule must not silently approximate')
assert len(list(artifacts.glob('*.html'))) == 2
assert json.loads((artifacts / 'cdm-margin-audit.json').read_text())['restated']['coverage_cents'] == 40_000_000
print('All scenario checks passed.')


All scenario checks passed.


## What to extend next

- Version inventory, prices, obligations, and schedules as well as ratings; bind all extraction to a coherent intended history.
- Replace the narrow schedule adapter with a tested translation of a documented CDM subset and reject unsupported criteria. Compare selected cases against authoritative CDM function outputs.
- Add issuer/group concentration using supplied exposure measures and GLEIF consolidation relationships (not beneficial ownership).
- Add authenticated generation/publication adapters explicitly; the custom bridge here must not be relabeled as that contract.
- Add temporal range reasoning, multiple independent sources, correction/retraction parity, and scale measurements before claiming incremental or performance benefits.

### Resources

- [Use-case proposal #409](https://github.com/mylonasc/gestaltdb/issues/409)
- [FINOS CDM collateral representation](https://cdm.finos.org/docs/eligible-collateral-representation) and [CDM execution/distribution boundaries](https://cdm.finos.org/docs/cdm-overview)
- [GestaltDB rc2 release notes](https://github.com/mylonasc/gestaltdb/blob/v0.14.0rc2/CHANGELOG.md)
- [Typed graph projection](https://github.com/mylonasc/gestaltdb/blob/v0.14.0rc2/docs/ontology-graph.rst), [structural validation](https://github.com/mylonasc/gestaltdb/blob/v0.14.0rc2/docs/reasoning-validation.rst), [proofs](https://github.com/mylonasc/gestaltdb/blob/v0.14.0rc2/docs/reasoning-proofs.rst), and [scoped negation](https://github.com/mylonasc/gestaltdb/blob/v0.14.0rc2/docs/reasoning-negation.rst)

## Cleanup
Copy any artifacts you want to retain **before** running the final cell. It closes the database and deletes only this notebook's disposable workspace. To rerun, start at the top; the history cells are intentionally sequential.

In [16]:
graph.close()
workspace.cleanup()
del graph, workspace
print('Closed LMDB and removed the disposable workspace.')


Closed LMDB and removed the disposable workspace.
